# 03 — Yakuniy model va submission
Bu notebook `src/train.py` dagi pipeline'ni **aynan** ishga tushiradi (kod takrorlanmaydi), shuning uchun natija skript bilan bir xil.

Pipeline:
1. `features_windows` — vaqt oynalari bo'yicha agregatlar (387 feature)
2. `features_histograms` — summa taqsimoti shakli (149 feature)
3. `features_relative` — tur / yo'nalish orasidagi nisbiy farqlar (317 feature)
4. LightGBM gain bo'yicha top-60 feature tanlash
5. 5-fold CV hisoboti
6. LightGBM (1500 daraxt × 10 seed) 0.7 + Logistik regressiya 0.3 rank-blend
7. `outputs/team_<ID>.csv`

Talablar: `pip install -r ../requirements.txt`. Ish vaqti ≈ 5–10 daqiqa, RAM ≈ 5 GB.

In [1]:
TEAM_ID = '7638B6E8'
import sys, os, time
os.chdir('..')                       # loyiha ildizidan ishlaymiz: data/, outputs/, src/
sys.path.insert(0, 'src')
import train
print(open('src/train.py').read().split('"""')[1])   # skript docstring'i


Fintech track — alert escalation probability
Pipeline:  transactions ──> per-signal features ──> feature selection ──> LightGBM + LogReg blend ──> CSV

Usage (run from the project root):
    python src/train.py --team 1234                 # full run: CV + submission -> outputs/team_1234.csv
    python src/train.py --team 1234 --quick         # skip the extra CV pass
    python src/train.py --team 1234 --data data     # data folder (default: data)

Requirements:
    pip install pandas pyarrow numpy scikit-learn lightgbm scipy

Expected output (5-fold CV): LightGBM ≈ 0.66, LogReg ≈ 0.65, blend ≈ 0.66 ROC-AUC.
Runtime ≈ 10 min, peak RAM ≈ 5 GB.

Outputs (in outputs/):
    team_<ID>.csv              submission
    train_log.txt              this run's log
    run_summary.json           params, feature counts, CV scores (read by src/site_data.py)
    selected_features.txt      the features the final model uses
    feature_importance.csv     LightGBM gain for every candidate feature
    oof

## Sozlamalar

In [2]:
print({k: getattr(train, k) for k in ['N_FOLDS', 'N_TOP_FEATURES', 'N_SEEDS_FINAL', 'BLEND_W_LGB', 'LGB_N_ESTIMATORS']})
print(train.LGB_PARAMS)

{'N_FOLDS': 5, 'N_TOP_FEATURES': 60, 'N_SEEDS_FINAL': 10, 'BLEND_W_LGB': 0.7, 'LGB_N_ESTIMATORS': 1500}
{'learning_rate': 0.005, 'num_leaves': 7, 'min_child_samples': 100, 'subsample': 0.7, 'subsample_freq': 1, 'colsample_bytree': 0.3, 'reg_lambda': 10, 'reg_alpha': 1, 'verbose': -1}


## Pipeline'ni ishga tushirish

In [3]:
train.TEAM_ID = TEAM_ID
t = time.time()
train.main(quick=False)
print(f'\njami {time.time()-t:.0f} s')

[17:25:09] train (14000, 3), test (6000, 2), escalation rate 0.1718
[17:25:09] building train features
[17:25:27]   features_windows        387 features  (18s)
[17:25:40]   features_histograms     149 features  (13s)
[17:28:18]   features_relative       317 features  (158s)
[17:28:18] building test features
[17:28:27]   features_windows        387 features  (9s)
[17:28:32]   features_histograms     148 features  (5s)
[17:29:25]   features_relative       317 features  (53s)
[17:29:25] feature matrix (14000, 853)
[17:29:49] all-feature LightGBM CV AUC = 0.6520
[17:29:49] top-10 features: grp_bank_otkazmasi_chiqim_minus_karta_chiqim_q75, grp_bank_otkazmasi_chiqim_minus_naqd_kirim_q75, all_amt_min, type_bank_otkazmasi_minus_naqd_q75, grp_bank_otkazmasi_kirim_minus_naqd_kirim_q75, grp_bank_otkazmasi_chiqim_minus_naqd_kirim_mean, grp_bank_otkazmasi_kirim_minus_naqd_kirim_mean, grp_bank_otkazmasi_chiqim_minus_karta_chiqim_mean, type_bank_otkazmasi_minus_karta_std, grp_bank_otkazmasi_kirim_min

## Natijalarni tekshirish

In [4]:
import pandas as pd
print(open('outputs/cv_report.txt').read())
sub = pd.read_csv(f'outputs/team_{TEAM_ID}.csv'); te = pd.read_csv('data/test_signals.csv')
checks = {
    'qatorlar soni = test signallar soni': len(sub) == len(te),
    'ustunlar: signal_id, ehtimollik': list(sub.columns) == ['signal_id', 'ehtimollik'],
    'signal_id lar takrorlanmaydi': sub.signal_id.is_unique,
    'barcha test ID lar mavjud': set(sub.signal_id) == set(te.signal_id),
    'bo\'sh qiymat yo\'q': sub.ehtimollik.notna().all(),
    '0 <= ehtimollik <= 1': sub.ehtimollik.between(0, 1).all(),
}
for k, v in checks.items(): print(('OK  ' if v else 'XATO'), k)
assert all(checks.values())
display(sub.head()); print(sub.ehtimollik.describe().round(3))

5-fold stratified CV, 60 selected features
all-feature LightGBM (early stopping): 0.6520
LightGBM top-60:                0.6610
LogReg top-60:                  0.6470
rank blend (0.7/0.3):                 0.6602

OK   qatorlar soni = test signallar soni
OK   ustunlar: signal_id, ehtimollik
OK   signal_id lar takrorlanmaydi
OK   barcha test ID lar mavjud
OK   bo'sh qiymat yo'q
OK   0 <= ehtimollik <= 1


,signal_id,ehtimollik
0,SG_000001,0.290183
1,SG_000007,0.378983
2,SG_000009,0.119367
3,SG_000010,0.414017
4,SG_000011,0.601417


count    6000.000
mean        0.500
std         0.282
min         0.001
25%         0.258
50%         0.495
75%         0.746
max         0.998
Name: ehtimollik, dtype: float64


In [5]:
feats = open('outputs/selected_features.txt').read().split()
print(len(feats), 'ta tanlangan feature; birinchi 15 tasi:'); print('\n'.join(feats[:15]))

60 ta tanlangan feature; birinchi 15 tasi:
grp_bank_otkazmasi_chiqim_minus_karta_chiqim_q75
grp_bank_otkazmasi_chiqim_minus_naqd_kirim_q75
all_amt_min
type_bank_otkazmasi_minus_naqd_q75
grp_bank_otkazmasi_kirim_minus_naqd_kirim_q75
grp_bank_otkazmasi_chiqim_minus_naqd_kirim_mean
grp_bank_otkazmasi_kirim_minus_naqd_kirim_mean
grp_bank_otkazmasi_chiqim_minus_karta_chiqim_mean
type_bank_otkazmasi_minus_karta_std
grp_bank_otkazmasi_kirim_minus_karta_chiqim_std
nb_amt_min
grp_bank_otkazmasi_kirim_minus_karta_chiqim_q75
grp_bank_otkazmasi_chiqim_minus_karta_chiqim_std
grp_karta_chiqim_minus_karta_kirim_std
type_bank_otkazmasi_minus_naqd_mean
